# 10m 도착 실험 · Bearing-UAV / iMatch R1 / R1+R4
Kaggle GPU와 Internet을 켜고 Run All을 실행하세요. 같은 800m 경로 4개, 시드 2개, 카메라, 실제 이동 오차를 세 방식에 적용합니다. 추정 거리 10m 이내에서 정지하고 그 순간 실제 거리도 10m 이내일 때만 성공입니다. 모든 방식을 처음부터 다시 실행합니다.

Bearing은 공식 cross_view 위치 회귀를 한 프레임당 한 번 실행합니다. 다음 블록을 고르는 이동 예측에는 직전 위치 추정과 기체에 보낸 이동 명령을 사용합니다. 실제 이동량과 시뮬레이터의 실제 방위는 전달하지 않습니다. iMatch는 기존 영상 이동 추정과 네 방향 매칭을 유지합니다. 결과는 각 방식의 항법 파이프라인 전체 비교입니다. 설치와 가중치 준비는 주행 시간에서 제외됩니다.

In [ ]:
from pathlib import Path
import datetime, json, subprocess, sys, time, shutil
import numpy as np, pandas as pd, cv2, torch, requests
from IPython.display import display, FileLink
if not torch.cuda.is_available(): raise RuntimeError('Enable Kaggle GPU')
WORK=Path('/kaggle/working/ab_bearing_10m')
RUN=WORK/datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SOURCE=RUN/'source'
SOURCE.mkdir(parents=True)
routes=[[[512,768],[1312,768]],[[1536,768],[896,1248]],[[1536,1536],[1056,896]],[[512,1536],[992,896]]]
config=dict(routes=routes,seeds=[2026,2027],radius_m=10,step_m=25,max_steps=64,
            bias_sd_deg=3,heading_sd_deg=2,length_sd=.05,side=2048,
            bbox=[478000,4976000,480048,4978048],crs='EPSG:26915',
            reference_layer='fsa2017',camera_layer='fsa2023')
for route in routes:
    points=np.asarray(route,float)
    assert points.shape==(2,2) and np.all((points>=185)&(points<config['side']-185))
    assert np.linalg.norm(points[0]-points[1])>config['radius_m']
(RUN/'config.json').write_text(json.dumps(config,indent=2))

In [ ]:
import hashlib
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
endpoint='https://imageserver.gisdata.mn.gov/cgi-bin/wms'
session=requests.Session()
session.mount('https://',HTTPAdapter(max_retries=Retry(total=3,backoff_factor=1,status_forcelist=[429,500,502,503,504])))
maps=WORK/'maps'
maps.mkdir(exist_ok=True)
metadata={}
for role in ('reference','camera'):
    layer=config[role+'_layer']
    params=dict(service='WMS',version='1.1.1',request='GetMap',layers=layer,styles='',srs=config['crs'],
                bbox=','.join(map(str,config['bbox'])),width=config['side'],height=config['side'],format='image/jpeg')
    response=session.get(endpoint,params=params,timeout=(20,120))
    response.raise_for_status()
    im=cv2.imdecode(np.frombuffer(response.content,np.uint8),cv2.IMREAD_COLOR)
    if im is None or im.shape!=(2048,2048,3) or im.std()<4: raise RuntimeError(f'Invalid WMS imagery: {layer}')
    digest=hashlib.sha256(response.content).hexdigest()
    path=maps/(digest+'.jpg')
    path.write_bytes(response.content)
    metadata[role]=dict(path=str(path),layer=layer,url=response.url,sha256=digest)
assert metadata['reference']['sha256']!=metadata['camera']['sha256']
(RUN/'maps.json').write_text(json.dumps(metadata,indent=2))

In [ ]:
def make_env(name, packages):
    import sys
    import importlib.metadata as metadata
    env=WORK/'envs'/name
    py=env/'bin/python'
    env.parent.mkdir(parents=True,exist_ok=True)
    log=RUN/(name+'_environment.log')

    def command(args, required=True):
        result=subprocess.run([str(x) for x in args],stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT,text=True)
        output=result.stdout or ''
        with log.open('a',encoding='utf-8') as f:
            f.write(repr([str(x) for x in args])+chr(10)+output+chr(10))
        if output: print(output,flush=True)
        if required and result.returncode:
            raise RuntimeError(f'Environment command failed ({result.returncode}). See {log}: '+output[-4000:])
        return result

    def valid_python():
        if not py.exists(): return False
        try:
            probe=command([py,'-c',
                'import sys; from pathlib import Path; '
                'assert Path(sys.prefix).resolve()==Path(sys.argv[1]).resolve(); '
                'assert sys.prefix != sys.base_prefix',env],required=False)
            return probe.returncode==0
        except OSError:
            return False

    if not valid_python():
        # Kaggle may omit ensurepip. Create the interpreter without invoking it.
        created=command([sys.executable,'-m','venv','--without-pip',
                         '--system-site-packages',env],required=False)
        if created.returncode or not valid_python():
            # Fallback also handles installations without the stdlib venv module.
            bootstrap=WORK/'virtualenv_bootstrap'
            command([sys.executable,'-m','pip','install','--target',bootstrap,'virtualenv'])
            launcher='import sys,runpy; sys.path.insert(0,sys.argv.pop(1)); runpy.run_module("virtualenv",run_name="__main__")'
            command([sys.executable,'-c',launcher,bootstrap,'--system-site-packages',env])
        if not valid_python():
            raise RuntimeError(f'Invalid environment interpreter: {py}. See {log}')
    # Parent pip can install into a pip-less environment; no ensurepip needed.
    pip_check=command([py,'-m','pip','--version'],required=False)
    if pip_check.returncode:
        command([sys.executable,'-m','pip','--python',py,'install','pip'])
    command([py,'-m','pip','--version'])
    pins=[package+'=='+metadata.version(package) for package in ('torch','torchvision','numpy')]
    constraint=env/'constraints.txt'
    constraint.write_text(chr(10).join(pins),encoding='utf-8')
    command([py,'-m','pip','install','-c',constraint,*packages])
    frozen=command([py,'-m','pip','freeze'])
    (RUN/(name+'_pip-freeze.txt')).write_text(frozen.stdout,encoding='utf-8')
    return py


PY=make_env('loma',['vismatch==1.3.1','scipy','rasterio'])

In [ ]:
sources={'imatch/geometry.py': '"""파이프라인 ④⑤: 대응점 검증(MAGSAC++)과 호모그래피 → 위치·heading.\n\n좌표 규칙 (Bearing-UAV와 같음)\n- 드론 영상 256 px, 지도 블록 512 px (2x2 타일). 블록 중심이 (0, 0), 1단위 = 128 px = 32 m.\n- 정답 변환: block_px = C + R(theta) (uav_px - 128),  R(t) = [[cos t, sin t], [-sin t, cos t]]\n"""\nimport numpy as np\n\nBLOCK_CENTER = 256.0\nPX_PER_UNIT = 128.0\nM_PER_UNIT = 32.0\n\nRANSAC_PX = 3.0  # MAGSAC++ 임계값(px). scripts/run.py --ransac-px 로 바꾼다\n\n\ndef set_ransac_px(px):\n    global RANSAC_PX\n    RANSAC_PX = float(px)\n\n\ndef rot(a):\n    t = np.deg2rad(a); c, s = np.cos(t), np.sin(t)\n    return np.array([[c, s], [-s, c]])\n\n\ndef turn(img, a):\n    """영상을 a도 돌린다 (반시계, cv2 규칙). 빈 모서리는 검은색."""\n    import cv2\n    h, w = img.shape[:2]\n    return cv2.warpAffine(img, cv2.getRotationMatrix2D((w / 2, h / 2), a, 1), (w, h), flags=cv2.INTER_LINEAR,\n                          borderMode=cv2.BORDER_CONSTANT, borderValue=0)\n\n\ndef wrap(d):\n    """각도 차이의 절댓값 (0~180도)."""\n    return np.abs((np.asarray(d) + 180) % 360 - 180)\n\n\ndef magsac(p0, p1):\n    """대응점 → (p0, p1, 호모그래피 H, inlier 마스크). 점이 4개 미만이면 H = None."""\n    import cv2\n    if len(p0) < 4:\n        return p0, p1, None, np.zeros(len(p0), bool)\n    H, mask = cv2.findHomography(p0, p1, cv2.USAC_MAGSAC, RANSAC_PX, maxIters=2000, confidence=0.995)\n    return p0, p1, H, (mask[:, 0].astype(bool) if mask is not None else np.zeros(len(p0), bool))\n\n\ndef pose_from_homography(H, scale, angle, uav_center):\n    """돌린 드론 영상 → 블록 호모그래피에서 위치(x, y: 블록 단위)와 heading(도), 축척을 구한다.\n\n    위치: 드론 영상 중심을 블록으로 옮긴 점. heading: 중심에서의 국소 선형 변환 L의 회전 성분.\n    회전 탐색으로 영상을 angle만큼 돌렸으므로 L에 R(angle)을 곱해 원래 영상 기준으로 되돌린다.\n    """\n    S = np.diag([scale, scale, 1.])\n    Hb = np.linalg.inv(S) @ H @ S                                   # 축척 보정 (base px 기준)\n    h = Hb @ np.array([uav_center[0], uav_center[1], 1.]); c = h[:2] / h[2]  # 회전해도 중심은 그대로\n    L = (Hb[:2, :2] - np.outer(c, Hb[2, :2])) / h[2] @ rot(angle)   # 중심에서의 야코비안\n    return dict(px=float((c[0] - BLOCK_CENTER) / PX_PER_UNIT), py=float((c[1] - BLOCK_CENTER) / PX_PER_UNIT),\n                head=float(np.degrees(np.arctan2(L[0, 1], L[0, 0]))), scale_est=float(np.sqrt(abs(np.linalg.det(L)))))\n', 'imatch/matchers.py': '"""파이프라인 ②③: 특징 추출과 매칭. 모든 매처는 (드론 영상, 블록) → magsac() 결과를 돌려준다.\n\n새 매처를 실험하려면 __call__(im0, im1) 을 가진 클래스를 만들고 get_matcher()에 이름을 추가하면 된다.\n"""\nimport os\n\nimport cv2\nimport numpy as np\n\nfrom .geometry import magsac\n\n# vismatch의 get_matcher에 없는 LoMa 변형은 직접 만든다 (\'loma\' = LoMa-B는 get_matcher로)\nLOMA_ARCH = {\'loma-b128\': \'LoMa-B128\', \'loma-l\': \'LoMa-L\', \'loma-g\': \'LoMa-G\'}\n\nclass VisMatch:\n    """vismatch 라이브러리의 공개 매처 (loma, romav2, sift-lightglue 등). GPU 권장.\n\n    LoMa 변형: \'loma\'(= LoMa-B, 주력), \'loma-b128\'(경량), \'loma-l\', \'loma-g\', \'loma-r\'.\n    """\n\n    def __init__(self, name, device=\'cuda\'):\n        import torch\n        self.name, self.torch = name, torch\n        if name in LOMA_ARCH:\n            from vismatch.im_models import loma\n            self.m = loma.LoMaMatcher(device, 2048, arch=LOMA_ARCH[name])\n        else:\n            from vismatch import get_matcher\n            self.m = get_matcher(name, device=device)\n        if name.startswith(\'loma\') and os.environ.get(\'LOMA_FP32\'):\n            # GTX 1650은 fp16에서 키포인트가 전부 NaN이 되어 오류 없이 짝 0개가 나온다. LOMA_FP32=1로 fp32를 쓴다.\n            # T4 결과(fp16)와 조금 다를 수 있으니 보고용 숫자는 T4 기본 설정으로 낸다\n            from vismatch.utils import force_float32\n            force_float32(self.m.matcher)\n        if name == \'romav2\':\n            # 기본 \'precise\' 설정은 T4(bf16 없음)에서 메모리 부족으로 멈췄다. 우리 영상은 256 / 512 px라\n            # \'base\' 설정(640 px, 단방향, fp32)을 쓴다.\n            from vismatch.im_models import romav2 as rv2\n            rv2._disable_autocast()\n            self.m.romav2_model.apply_setting(os.environ.get(\'ROMAV2_SETTING\', \'base\'))\n            self.name = f"romav2-{os.environ.get(\'ROMAV2_SETTING\', \'base\')}"\n\n    def __call__(self, im0, im1):\n        t = lambda im: self.torch.from_numpy(im).permute(2, 0, 1).float() / 255.\n        r = self.m(t(im0), t(im1))\n        p0, p1 = r[\'matched_kpts0\'], r[\'matched_kpts1\']\n        # vismatch는 inlier 점만 주고 마스크는 안 주므로 같은 점으로 MAGSAC++를 다시 돌린다\n        return magsac(np.float32(p0).reshape(-1, 2), np.float32(p1).reshape(-1, 2))\n\n\nclass CvSift:\n    """GPU 없이 도는 기준선: OpenCV SIFT + Lowe 비율 검사 + MAGSAC++. 설치·동작 확인용."""\n    name = \'cv-sift\'\n\n    def __init__(self):\n        self.sift = cv2.SIFT_create(4096); self.bf = cv2.BFMatcher()\n\n    def __call__(self, im0, im1):\n        k0, d0 = self.sift.detectAndCompute(cv2.cvtColor(im0, cv2.COLOR_RGB2GRAY), None)\n        k1, d1 = self.sift.detectAndCompute(cv2.cvtColor(im1, cv2.COLOR_RGB2GRAY), None)\n        if d0 is None or d1 is None or len(k0) < 2 or len(k1) < 2:\n            return np.empty((0, 2)), np.empty((0, 2)), None, np.zeros(0, bool)\n        mm = [m for m, n in self.bf.knnMatch(d0, d1, k=2) if m.distance < 0.8 * n.distance]\n        p0 = np.float32([k0[m.queryIdx].pt for m in mm]).reshape(-1, 2)\n        p1 = np.float32([k1[m.trainIdx].pt for m in mm]).reshape(-1, 2)\n        return magsac(p0, p1)\n\n\nclass VggMNN:\n    """비교 실험 \'VGG16 백본은 그대로 두면?\' (docs/02_실험기록.md 7장). 원 Bearing-UAV 체크포인트의 고정\n    VGG16 conv4_3(B2가 상관을 계산하는 그 특징)을 점 단위 기술자로 쓴다.\n\n    특징 맵의 칸(stride 8, 중심 8 i + 4)이 점이다. 블록 평균을 뺀 코사인 유사도로 상호 최근접 짝을 짓고,\n    이후는 다른 매처와 같은 MAGSAC++ / 호모그래피를 쓴다. 회전으로 생긴 검은 모서리의 칸은 뺀다.\n    capstone1 저장소와 체크포인트가 필요하다:\n      CAPSTONE1_DIR=<capstone1 경로> VGG_CKPT=Bearing_UAV/cross_view python scripts/run.py --matchers vgg-mnn --device cpu\n    결과: Singapore 44%(B2 69%, 100샘플) → 원 모델 특징으로는 점 단위 매칭이 안 된다.\n    """\n    name = \'vgg-mnn\'\n\n    def __init__(self):\n        import sys\n        import torch\n        root = os.environ[\'CAPSTONE1_DIR\']\n        sys.path.insert(0, root)\n        from diag.diag_modules import DEV, load_model, to_tensor\n        self.torch, self.tt, self.dev = torch, to_tensor, DEV\n        self.bb = load_model(os.path.join(root, os.environ.get(\'VGG_CKPT\', \'Bearing_UAV/cross_view\'))).backbone\n\n    def _feat(self, im):\n        with self.torch.no_grad():\n            f = self.bb(self.tt(im).unsqueeze(0).to(self.dev))[0]          # [C, h, w]\n        C, h, w = f.shape; st = im.shape[0] / h\n        yy, xx = np.mgrid[0:h, 0:w]\n        pts = np.stack([xx.ravel() * st + st / 2, yy.ravel() * st + st / 2], 1).astype(np.float32)\n        return f.reshape(C, -1).T, pts\n\n    def __call__(self, im0, im1):\n        F = self.torch.nn.functional\n        d0, p0 = self._feat(im0); d1, p1 = self._feat(im1)\n        valid = im0[np.clip(p0[:, 1].astype(int), 0, im0.shape[0] - 1), np.clip(p0[:, 0].astype(int), 0, im0.shape[1] - 1)].sum(1) > 0\n        d0, p0 = d0[self.torch.from_numpy(valid)], p0[valid]\n        mu = d1.mean(0, keepdim=True)\n        d0, d1 = F.normalize(d0 - mu, dim=1), F.normalize(d1 - mu, dim=1)\n        S = d0 @ d1.T\n        j = S.argmax(1); i = S.argmax(0)\n        mutual = (i[j] == self.torch.arange(len(j), device=j.device)).cpu().numpy()\n        j = j.cpu().numpy()\n        return magsac(p0[mutual], p1[j[mutual]])\n\n\ndef get_matcher(name, device=\'cuda\'):\n    if name == \'cv-sift\':\n        return CvSift()\n    if name == \'vgg-mnn\':\n        return VggMNN()\n    return VisMatch(name, device)\n', 'imatch/pipeline.py': '"""파이프라인 ①~⑤를 한 샘플에 적용한다.\n\n① 회전 탐색: 드론 영상을 angles의 각 각도로 돌린다 (angles=[0]이면 탐색 없음, 원 Bearing-UAV와 같은 조건)\n②③ 매칭: matcher(돌린 드론 영상, 블록)\n④ 검증: MAGSAC++ inlier\n⑤ 위치·heading: 호모그래피 → pose_from_homography\n→ inlier가 가장 많은 각도의 결과를 고른다.\n   stop_inl > 0이면 각도를 차례로 해 보다가 inlier가 stop_inl 이상인 각도가 나오면 거기서 멈춘다 (조기 종료, 빠르게).\n"""\nimport time\n\nimport cv2\nimport numpy as np\n\nfrom .geometry import pose_from_homography, rot, turn\n\n\ndef localize(matcher, uav, block, gt, scale=1.0, angles=(0.0,), px_tol=8.0, stop_inl=0):\n    """각도별 기록 목록과 고른 결과(best, 없으면 None)를 돌려준다.\n\n    gt = (x_norm, y_norm, theta): 위치 추정에는 쓰지 않고, 대응점이 맞는지(good_*) 채점하는 데만 쓴다.\n    """\n    hu, wu = uav.shape[:2]; cu = np.array([wu / 2, hu / 2])\n    U = cv2.resize(uav, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC) if scale != 1 else uav\n    B = cv2.resize(block, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC) if scale != 1 else block\n    xn, yn, th = gt; C = np.array([256 + xn * 128, 256 + yn * 128]); Rg = rot(th)\n    recs = []\n    for a in angles:\n        t0 = time.time()\n        p0, p1, H, inl = matcher(turn(U, a), B)\n        dt = time.time() - t0\n        p0 = np.asarray(p0, np.float64) / scale; p1 = np.asarray(p1, np.float64) / scale\n        # 원래 드론 영상 좌표로 되돌림: p_turned = cu + R(a) (p - cu)  ->  p = cu + R(a)^T (p_turned - cu)\n        po = (p0 - cu) @ rot(a) + cu if len(p0) else p0\n        good = np.linalg.norm(C + (po - cu) @ Rg.T - p1, axis=1) < px_tol if len(p0) else np.zeros(0, bool)\n        r = dict(angle=a, n_match=int(len(p0)), n_inl=int(inl.sum()), sec=dt,\n                 good_match=float(good.mean()) if len(good) else np.nan,\n                 good_inl=float(good[inl].mean()) if inl.sum() else np.nan)\n        if H is not None and inl.sum() >= 4:\n            r.update(pose_from_homography(H, scale, a, cu))\n        recs.append(r)\n        if stop_inl > 0 and \'px\' in r and r[\'n_inl\'] >= stop_inl:\n            break\n    ok = [r for r in recs if \'px\' in r]\n    best = max(ok, key=lambda r: r[\'n_inl\']) if ok else None\n    return recs, best\n', 'imatch/__init__.py': '"""Trone 이미지 매칭 파이프라인: 드론 영상 + 지도 블록 → 위치·heading (학습 없음).\n\n  geometry.py  ④⑤ MAGSAC++ 검증, 호모그래피 → 위치·heading\n  matchers.py  ②③ 특징 추출·매칭 (LoMa, RoMa v2, SIFT ...)\n  pipeline.py  ①~⑤ 한 샘플 처리 (회전 탐색 포함)\n  blocks.py    ⑥ 후보 블록 자르기, 이웃 블록 좌표\n  nav.py       1 km 닫힌 루프 항법 (2D 크롭, 3D 렌더 입력)\n  metrics.py   팀 공통 평가 규칙\n"""\ndef __getattr__(name):\n    # CPU evaluation should not require GPU/image-processing dependencies.\n    if name == \'get_matcher\':\n        from .matchers import get_matcher\n        return get_matcher\n    if name == \'localize\':\n        from .pipeline import localize\n        return localize\n    raise AttributeError(name)\n', 'map_cache.py': 'class StepMapMatcher:\n    """Instrument both variants. Reuse only the second image of the pinned LoMa pair API."""\n    def __init__(self, matcher, reuse):\n        self.matcher = matcher\n        self.reuse = reuse\n        self.block = self.features = None\n        self.hits = self.extractions = 0\n        model = matcher.m.matcher\n        self.extract = model.detect_and_describe\n        model.detect_and_describe = self.describe\n\n    def __call__(self, uav, block):\n        if block is not self.block:\n            self.block = block\n            self.features = None\n        self.image_index = 0\n        result = self.matcher(uav, block)\n        if self.image_index != 2:\n            raise RuntimeError(\'Expected UAV and map extraction once per LoMa pair\')\n        return result\n\n    def describe(self, image, count):\n        self.image_index += 1\n        if self.image_index == 1:\n            return self.extract(image, count)\n        if self.image_index != 2:\n            raise RuntimeError(\'Unexpected feature extraction order\')\n        if self.reuse and self.features is not None:\n            self.hits += 1\n            return self.features\n        self.extractions += 1\n        result = self.extract(image, count)\n        if self.reuse:\n            self.features = (result[0], result[1], None, None)\n        return result\n', 'ab.py': "import json,sys,time,random\nfrom pathlib import Path\nimport numpy as np,pandas as pd,cv2,torch\nfrom imatch.geometry import turn\nfrom imatch.matchers import get_matcher\nfrom map_cache import StepMapMatcher\ndef locate_r1_r4(guess, streak, available, match, recover):\n    center = tuple(np.rint(guess / 256).astype(int))\n    attempts = 0\n    def probe(key, threshold, inside):\n        nonlocal attempts\n        if key not in available:\n            return None\n        records, best = match(key)\n        attempts += len(records)\n        if best is None or best['n_inl'] < threshold:\n            return None\n        offset = np.array([best['px'], -best['py']])\n        if inside and np.abs(offset).max() > 1.1:\n            return None\n        return np.array(key)*256 + offset*128, best['basis']\n\n    position = probe(center, 15, False)\n    if position is not None:\n        return (*position, 0, attempts, 'r1', False)\n    streak += 1\n    if not recover:\n        return guess, None, streak, attempts, 'guess', False\n    wide = streak >= 5\n    candidates = []\n    for dx in (-1, 0, 1):\n        for dy in (1, 0, -1):  # northing increases opposite to map row\n            if (dx,dy)==(0,0) or (not wide and abs(dx)+abs(dy)>1):\n                continue\n            position = probe((center[0]+dx, center[1]+dy), 20 if wide else 15, True)\n            if position is not None:\n                candidates.append(position)\n    if candidates:\n        return (*min(candidates,key=lambda p:np.linalg.norm(p[0]-guess)), 0, attempts, 'r4', wide)\n    return guess, None, streak, attempts, 'guess', wide\n\n\ndef movement(previous,current):\n    pts=cv2.goodFeaturesToTrack(previous,400,.01,5)\n    if pts is None or len(pts)<8: return None\n    nxt,status,_=cv2.calcOpticalFlowPyrLK(previous,current,pts,None)\n    if nxt is None: return None\n    ok=status.ravel().astype(bool)\n    if ok.sum()<8: return None\n    # Current pixels -> previous pixels, suitable for propagating a map transform.\n    matrix,mask=cv2.estimateAffinePartial2D(nxt[ok],pts[ok],method=cv2.RANSAC,ransacReprojThreshold=2)\n    if matrix is None or mask.sum()<8 or not np.isfinite(matrix).all(): return None\n    return matrix\n\ncenter=np.array([128.,128.])\ndef advance(est,basis,motion):\n    if motion is None or basis is None: return est,None\n    shift=motion[:,:2]@center+motion[:,2]-center\n    return est+basis@shift,basis@motion[:,:2]\n\ndef probe_angles(matcher,q,block):\n    records=[];best=None\n    for angle in (0,90,180,270):\n        _,_,H,mask=matcher(turn(q,angle),block)\n        count=int(mask.sum());records.append(count)\n        if H is None or count<4: continue\n        rot=np.vstack([cv2.getRotationMatrix2D(tuple(center),angle,1),[0,0,1]])\n        T=H@rot\n        v=T@np.r_[center,1.]\n        if abs(v[2])<1e-9: continue\n        xy=v[:2]/v[2]\n        jac=(T[:2,:2]-np.outer(xy,T[2,:2]))/v[2]\n        if not np.isfinite(jac).all() or not np.isfinite(xy).all(): continue\n        result=dict(n_inl=count,px=(xy[0]-256)/128,py=(xy[1]-256)/128,\n                    basis=np.diag([1.,-1.])@jac)\n        if best is None or count>best['n_inl']: best=result\n        if count>=15: break\n    return records,best\n\n\ndef render(camera, position, yaw):\n    # Output pixel (128,128) samples the aircraft position; at zero yaw north is up.\n    theta=np.deg2rad(yaw)\n    c,s=np.cos(theta),np.sin(theta)\n    axes=np.array([[c,-s],[s,c]])\n    middle=np.array([position[0],camera.shape[0]-position[1]])\n    matrix=np.column_stack((axes,middle-axes@center))\n    return cv2.warpAffine(camera,matrix,(256,256),flags=cv2.INTER_LINEAR|cv2.WARP_INVERSE_MAP)\n\ndef replay(config, reference, camera, route, seed, matcher, recover, bearing=False):\n    start,goal=np.asarray(route,dtype=float)\n    actual=start.copy()\n    estimate=start.copy()\n    rng=np.random.default_rng(seed)\n    bias=rng.normal(0,config['bias_sd_deg'])\n    noises=rng.normal(size=(config['max_steps'],2))\n    side=config['side']\n    tiles={(x,y) for x in range(1,side//256) for y in range(1,side//256)}\n    basis=prev=None\n    streak=0\n    traveled=0.\n    yaw=0.\n    rows=[]\n    outcome='timeout'\n    for k in range(config['max_steps']+1):\n        if np.any(actual<182) or np.any(actual>=side-182):\n            outcome='out_of_map'\n            break\n        q=render(camera,actual,yaw)\n        if bearing:\n            gray=None\n            vo_used=False\n            new_basis=None\n            key=tuple(np.rint(estimate/256).astype(int))\n            attempts=int(key in tiles)\n            update=estimate\n            used='guess'\n            wide=False\n            if attempts:\n                col,row=key[0]*256,side-key[1]*256\n                prediction=matcher.predict(q,reference[row-256:row+256,col-256:col+256])\n                offset=np.array([prediction['x'],-prediction['y']])\n                if prediction['valid'] and np.isfinite(offset).all():\n                    update=np.asarray(key)*256 + 128*offset\n                    used='bearing'\n        else:\n            gray=cv2.cvtColor(q,cv2.COLOR_RGB2GRAY)\n            motion=movement(prev,gray) if prev is not None else None\n            vo_used=basis is not None and motion is not None\n            if prev is not None: estimate,basis=advance(estimate,basis,motion)\n            def match(key):\n                col,row=key[0]*256,side-key[1]*256\n                tile=reference[row-256:row+256,col-256:col+256]\n                return probe_angles(matcher,q,tile)\n            update,new_basis,streak,attempts,used,wide=locate_r1_r4(estimate,streak,tiles,match,recover and k>0)\n        if new_basis is not None: basis=new_basis\n        if k: estimate=update\n        else: streak=0\n        delta=goal-estimate\n        remaining=np.linalg.norm(delta)\n        rows.append(dict(step=k,actual_x=actual[0],actual_y=actual[1],est_x=estimate[0],est_y=estimate[1],\n                         used=used,wide=wide,vo_used=vo_used,attempts=attempts,estimated_goal_m=remaining,\n                         true_goal_m=np.linalg.norm(goal-actual)))\n        if remaining<=config['radius_m']:\n            outcome='arrived' if np.linalg.norm(goal-actual)<=config['radius_m'] else 'false_stop'\n            break\n        if k==config['max_steps']: break\n        direction=delta/remaining\n        heading_error=np.deg2rad(bias+noises[k,0]*config['heading_sd_deg'])\n        c,s=np.cos(heading_error),np.sin(heading_error)\n        stride=min(config['step_m'],remaining)*max(0.,1+noises[k,1]*config['length_sd'])\n        displacement=stride*np.array([[c,-s],[s,c]])@direction\n        if bearing: estimate=estimate+direction*min(config['step_m'],remaining)\n        actual+=displacement\n        traveled+=np.linalg.norm(displacement)\n        # The simulated camera follows the motion heading. The controller receives only its image.\n        yaw=np.degrees(np.arctan2(displacement[0],displacement[1]))\n        prev=gray\n    return rows,dict(outcome=outcome,success=outcome=='arrived',terminal_m=np.linalg.norm(goal-actual),\n                     distance_m=traveled,commands=k,recoveries=sum(r['used']=='r4' for r in rows))\n\ndef main():\n    run=Path(sys.argv[1]);variant=sys.argv[2]\n    assert variant in ('bearing','r1','r1_r4')\n    config=json.loads((run/'config.json').read_text())\n    meta=json.loads((run/'maps.json').read_text())\n    def read(role): return cv2.cvtColor(cv2.imread(meta[role]['path']),cv2.COLOR_BGR2RGB)\n    reference,camera=read('reference'),read('camera')\n    random.seed(2026);np.random.seed(2026);torch.manual_seed(2026);cv2.setRNGSeed(2026)\n    if variant=='bearing':\n        from bearing_adapter import BearingAdapter\n        prepared=json.loads((run/'Bearing-UAV_prepared.json').read_text())\n        base=BearingAdapter(Path(prepared['repo']),prepared['weights_dir'],'cuda:0')\n        assert base.metadata['checkpoint_sha256']==prepared['checkpoint_sha256']\n    else:\n        base=get_matcher('loma','cuda')\n    trials=[];logs=[]\n    with torch.inference_mode():\n        warm=render(camera,[512,768],0)\n        if variant=='bearing':\n            base.predict(warm,reference[:512,:512])\n            matcher=base\n        else:\n            base(warm,reference[:512,:512])\n            matcher=StepMapMatcher(base,reuse=True)\n        torch.cuda.synchronize()\n        for i,route in enumerate(config['routes']):\n            for seed in config['seeds']:\n                cv2.setRNGSeed(seed);torch.manual_seed(seed)\n                if variant!='bearing': matcher.features=matcher.block=None\n                torch.cuda.synchronize();t=time.perf_counter()\n                rows,stats=replay(config,reference,camera,route,seed,matcher,variant=='r1_r4',bearing=variant=='bearing')\n                torch.cuda.synchronize();stats['wall_s']=time.perf_counter()-t\n                trials.append(dict(route=i,seed=seed,**stats))\n                logs.extend(dict(route=i,seed=seed,**r) for r in rows)\n                print(variant,i,seed,stats,flush=True)\n    pd.DataFrame(trials).to_csv(run/f'{variant}_trials.csv',index=False)\n    pd.DataFrame(logs).to_csv(run/f'{variant}_steps.csv',index=False)\nif __name__=='__main__': main()\n", 'bearing_adapter.py': 'from pathlib import Path\nimport numpy as np\nimport torch\nfrom hashlib import sha256 as _sha256\ndef sha256(path):\n    h=_sha256()\n    with open(path,"rb") as f:\n        for b in iter(lambda:f.read(1048576),b""): h.update(b)\n    return h.hexdigest()\n\nclass BearingAdapter:\n    def __init__(self, repo, weights_dir, device):\n        import sys\n        sys.path.insert(0, str(repo))\n        # Official utils_transform accesses np.complex_ during import on NumPy 2.\n        for name, value in {\'complex_\': np.complex128, \'float_\': np.float64}.items():\n            if name not in np.__dict__:\n                setattr(np, name, value)\n        # imgaug 0.4 imports np.sctypes, removed in NumPy 2. This compatibility alias\n        # affects import-time augmentation setup; evaluation uses only ToTensor/Normalize.\n        if not hasattr(np, \'sctypes\'):\n            np.sctypes = {\'int\': [np.int8, np.int16, np.int32, np.int64],\n                          \'uint\': [np.uint8, np.uint16, np.uint32, np.uint64],\n                          \'float\': [np.float16, np.float32, np.float64],\n                          \'complex\': [np.complex64, np.complex128], \'others\': [bool, object, str, bytes]}\n        from cvphr.models.posaglreg.models import load_config_and_model\n        cls, kw = load_config_and_model(str(weights_dir))\n        self.model = cls(**kw)\n        path = Path(weights_dir) / \'best_model.pth\'\n        state = torch.load(path, map_location=\'cpu\', weights_only=True)\n        self.model.load_state_dict(state[\'model_state_dict\'], strict=True)\n        self.model = self.model.float().to(device).eval()\n        self.device, self.modules = device, [self.model]\n        self.metadata = {\'model\': \'Bearing-UAV\', \'variant\': \'official cross_view\',\n                         \'weights\': str(path), \'checkpoint_sha256\': sha256(path),\n                         \'training_regime\': \'public-pretrained: training-city overlap must be audited\'}\n\n    def predict(self, uav, block):\n        # Official p1,p2,p3,p4 order = TL,TR,BL,BR; fifth image = UAV.\n        patches = [block[y:y+256, x:x+256] for x, y in [(0, 0), (256, 0), (0, 256), (256, 256)]] + [uav]\n        x = np.stack(patches).astype(np.float32) / 255\n        x = (x - np.array([.485, .456, .406], np.float32)) / np.array([.229, .224, .225], np.float32)\n        x = torch.from_numpy(x.transpose(0, 3, 1, 2).copy())[None].to(self.device)\n        pos, _direction = self.model(x)\n        xy = pos[0].float().cpu().numpy()\n        return {\'x\': float(xy[0]), \'y\': float(xy[1]), \'valid\': True}\n'}
for name,source in sources.items():
    p=SOURCE/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_text(source,encoding='utf8')
def evaluate(mode):
    subprocess.run([str(BEARING_PY if mode=='bearing' else PY),'-u',str(SOURCE/'ab.py'),str(RUN),mode],check=True)

In [ ]:
CACHE=WORK/'pretrained'
CACHE.mkdir(exist_ok=True)
def save_json(p,v): p.write_text(json.dumps(v,indent=2))
from hashlib import sha256
def hash_file(p):
    h=sha256()
    with p.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
BEARING_WEIGHTS_DIR = ''  # 비우면 공식 cross_view 가중치를 다운로드
BEARING_REVISION = 'd16e558a14bd0a9142c6901793fe6a6813e4e954'
BEARING_PY=make_env('bearing',['huggingface_hub','imgaug==0.4.0','scikit-image','scipy'])

import zipfile
BEARING_REPO = CACHE / 'bearinguav'
BEARING_READY, BEARING_ERROR = False, ''
try:
    if not (BEARING_REPO / '.git').exists():
        subprocess.run(['git', 'clone', 'https://github.com/liukejia121/bearinguav.git', str(BEARING_REPO)], check=True)
    subprocess.run(['git', '-C', str(BEARING_REPO), 'checkout', BEARING_REVISION], check=True)
    bearing_commit = subprocess.check_output(['git', '-C', str(BEARING_REPO), 'rev-parse', 'HEAD'], text=True).strip()
    if not BEARING_WEIGHTS_DIR:
        archive = subprocess.check_output([str(BEARING_PY),'-c',
            "from huggingface_hub import hf_hub_download; print(hf_hub_download('HaoyZhou/bearinguav','Bearing_UAV.zip'))"],text=True).strip().splitlines()[-1]
        destination = CACHE / 'bearing_weights'
        destination.mkdir(exist_ok=True)
        with zipfile.ZipFile(archive) as z:
            for member in z.infolist():
                resolved = (destination / member.filename).resolve()
                if not resolved.is_relative_to(destination.resolve()):
                    raise ValueError('Unsafe ZIP path')
            z.extractall(destination)
        candidates = [p.parent for p in destination.rglob('best_model.pth') if p.parent.name == 'cross_view']
        if len(candidates) != 1:
            raise ValueError(f'Select BEARING_WEIGHTS_DIR explicitly: {candidates}')
        BEARING_WEIGHTS_DIR = str(candidates[0])
    if not (Path(BEARING_WEIGHTS_DIR) / 'training_configure.json').is_file():
        raise FileNotFoundError('training_configure.json is required alongside best_model.pth')
    BEARING_READY = True
except Exception as exc:
    BEARING_ERROR = repr(exc)
    print('Bearing-UAV preparation failed:', BEARING_ERROR)

if not BEARING_READY: raise RuntimeError(BEARING_ERROR)
save_json(RUN/'Bearing-UAV_prepared.json',dict(repo=str(BEARING_REPO),repo_commit=bearing_commit,
    weights_dir=BEARING_WEIGHTS_DIR,checkpoint_sha256=hash_file(Path(BEARING_WEIGHTS_DIR)/'best_model.pth')))
assert json.loads((RUN/'Bearing-UAV_prepared.json').read_text())['checkpoint_sha256']=='31dc15f9fa48cfcd5e58c22387ea5509ba932bb6f70be0d2dfc3c9bac7c45644'


In [ ]:
evaluate('bearing')

In [ ]:
evaluate('r1')

In [ ]:
evaluate('r1_r4')

In [ ]:
modes=['r1','r1_r4','bearing']
tables=[pd.read_csv(RUN/f'{m}_trials.csv').set_index(['route','seed'],verify_integrity=True) for m in modes]
assert all(t.index.equals(tables[0].index) for t in tables)
paired=pd.concat([t.add_suffix('_'+m) for m,t in zip(modes,tables)],axis=1)
for col in ('terminal_m','wall_s','distance_m'):
    for m in modes[1:]: paired[m+'_minus_r1_'+col]=paired[col+'_'+m]-paired[col+'_r1']
paired.to_csv(RUN/'paired_trials.csv')
summary=pd.DataFrame([dict(mode=m,trials=len(t),arrivals=int(t.success.sum()),success_pct=100*t.success.mean(),
    false_stops=(t.outcome=='false_stop').sum(),timeouts=(t.outcome=='timeout').sum(),out_of_map=(t.outcome=='out_of_map').sum(),
    terminal_median_m=t.terminal_m.median(),total_distance_m=t.distance_m.sum(),wall_s=t.wall_s.sum(),recoveries=t.recoveries.sum())
    for m,t in zip(modes,tables)])
summary.to_csv(RUN/'comparison.csv',index=False);display(summary);display(paired)
archive=shutil.make_archive(str(RUN)+'_results','zip',RUN)
display(FileLink(str(Path(archive).relative_to('/kaggle/working'))))